# 学习的技巧练习参考答案

本文件是配套练习题的参考答案，请先在手写练习后再对照。

# 学习的技巧练习

- 深度神经网络及其问题（梯度消失与梯度爆炸）
- 更新参数方法的优化（SGD 的缺点、Momentum、学习率衰减、AdaGrad / RMSProp / Adam）
- 参数初始化（常数、秩、正态、均匀、Xavier、He）
- 正则化（Batch Normalization、权值衰减、Dropout）
- 应用案例：房价预测

说明：本练习直接使用教材第 5 章的原始示例（目标函数、超参数、网络结构等均与教材一致），
请先阅读题目描述，再在下方代码单元格中**手写代码**完成练习，写完后与 `answer` 目录下的答案对照。

部分练习所需的数据位于项目根目录的 `data/` 下，本 notebook 中使用相对路径 `../../data/...`。

先执行下面的单元格导入所需的库。

In [ ]:
import torch
import numpy as np
import pandas as pd
import torch.nn as nn
import matplotlib.pyplot as plt
from torch import optim
from torch.utils.data import TensorDataset, DataLoader

np.set_printoptions(precision=4, suppress=True)
torch.manual_seed(0)

print("torch version:", torch.__version__)

## 5.1 深度神经网络及其问题

网络层数加深后，梯度在隐藏层反向传播时可能"消失"（越来越小）或"爆炸"（越来越大）。
一个直观的解释是：反向传播每经过一层，梯度都会乘上一个与权重尺度相关的系数，连乘的结果决定了梯度最终的尺度。

**练习 1**：深层网络中的梯度消失与梯度爆炸

反向传播每经过一层，梯度都会乘上一个与权重尺度相关的系数：权重整体偏小则梯度逐层衰减（梯度消失），
权重整体偏大则梯度逐层放大（梯度爆炸）。

要求：

1. 编写 `build_deep_net(depth=10, dim=32, std=1.0)`：用 `nn.Sequential` 连续堆叠 `depth` 个
   `nn.Linear(dim, dim)`（权重用 `nn.init.normal_(mean=0.0, std=std)` 初始化，偏置置 0），
   最后再接一个输出维度为 1 的线性层；
2. 先 `torch.manual_seed(0)`，用 `x = torch.randn(16, 32)`、`y = torch.randn(16, 1)` 前向计算 MSE 损失并 `.backward()`；
3. 分别在 `std = 0.5, 1.0, 1.5` 下构建网络（每次构建前都 `torch.manual_seed(0)`，保证只有 std 不同），
   打印损失值和**第 1 层权重**的梯度范数（科学计数法），观察随 std 增大的变化趋势。

提示：第 1 层就是 `net[0]`，梯度范数用 `net[0].weight.grad.norm().item()`。

In [ ]:
# 练习 1：深层网络中的梯度消失与梯度爆炸

# 练习 1：深层网络中的梯度消失与梯度爆炸


def build_deep_net(depth=10, dim=32, std=1.0):
    layers = []
    for _ in range(depth):
        linear = nn.Linear(dim, dim)
        nn.init.normal_(linear.weight, mean=0.0, std=std)
        nn.init.zeros_(linear.bias)
        layers.append(linear)
    layers.append(nn.Linear(dim, 1))
    return nn.Sequential(*layers)


torch.manual_seed(0)
x = torch.randn(16, 32)
y = torch.randn(16, 1)

for std in (0.5, 1.0, 1.5):
    torch.manual_seed(0)                                  # 保证只有 std 不同
    net = build_deep_net(depth=10, dim=32, std=std)
    loss = nn.MSELoss()(net(x), y)
    loss.backward()
    print(f"std={std:<4} loss={loss.item():.3e} "
          f"第 1 层权重梯度范数={net[0].weight.grad.norm().item():.3e}")

**练习 2**：手写链式法则并与 autograd 对比

考虑一个 3 层线性网络（无激活函数）：`x(2×5) → W1(4×5) → W2(3×4) → W3(1×3)`，
输出为 `out = x @ W1.T @ W2.T @ W3.T`，损失为 `L = sum(out ** 2)`。

要求：

1. 用 `torch.manual_seed(1)` 生成 `W1 = torch.randn(4, 5) * 0.5`、`W2 = torch.randn(3, 4) * 0.5`、
   `W3 = torch.randn(1, 3) * 0.5`、`x = torch.randn(2, 5)`；
2. 把三个权重各克隆一份并设置 `requires_grad=True`，前向计算损失后 `.backward()`，得到 autograd 的 `dL/dW1`；
3. 在 `torch.no_grad()` 下手写反向传播，求出 `dL/dW1`；
4. 比较两者范数与最大绝对误差，验证手写结果与 autograd 一致。

提示：`L = sum(out ** 2)` 对 `out` 的梯度是 `2 * out`；对 `y = h @ W.T`，有 `dL/dW = dL/dy.T @ h`、`dL/dh = dL/dy @ W`。

In [ ]:
# 练习 2：手写链式法则并与 autograd 对比

# 练习 2：手写链式法则并与 autograd 对比

torch.manual_seed(1)
W1 = torch.randn(4, 5) * 0.5
W2 = torch.randn(3, 4) * 0.5
W3 = torch.randn(1, 3) * 0.5
x = torch.randn(2, 5)

# 用 autograd 求梯度
w1 = W1.clone().requires_grad_(True)
w2 = W2.clone().requires_grad_(True)
w3 = W3.clone().requires_grad_(True)
out = x @ w1.t() @ w2.t() @ w3.t()
loss = (out ** 2).sum()
loss.backward()

# 手写链式法则
with torch.no_grad():
    h1 = x @ W1.t()          # 2x4
    h2 = h1 @ W2.t()         # 2x3
    out2 = h2 @ W3.t()       # 2x1
    dout = 2 * out2          # dL/dout
    dh2 = dout @ W3          # 2x3
    dh1 = dh2 @ W2           # 2x4
    dW1 = dh1.t() @ x        # 4x5

print("autograd dL/dW1 范数:", w1.grad.norm().item())
print("手写     dL/dW1 范数:", dW1.norm().item())
print("最大绝对误差:", (dW1 - w1.grad).abs().max().item())

## 5.2 更新参数方法的优化

SGD 简单经典，但在很多问题上并不高效。本节练习 Momentum、学习率衰减以及
AdaGrad / RMSProp / Adam 等改进方法。

为便于对比，本节统一使用教材示例的原始目标函数 $f(x_1,x_2)=0.05x_1^{2}+x_2^{2}$，
其矩阵形式为 `X ** 2 @ w`，其中 `w = torch.tensor([[0.05], [1.0]])`；起点统一取 `(-7.0, 2.0)`。
练习 3 的代码单元格中先定义 `w`，后续练习复用。

**练习 3**：编写通用的优化轨迹记录函数

为便于对比不同的参数更新方法，先准备一个工具函数：在目标函数上迭代更新自变量 `X`，并记录每一步的 `X`。

要求：先定义 `w = torch.tensor([[0.05], [1.0]])`，再编写 `gradient_descent(X, optimizer, n_iters)`：

- `X`：初始化好的自变量（`requires_grad=True`），**直接对传入的 `X` 迭代**，不要重新创建；
- 每次迭代：`y = X ** 2 @ w` → `y.backward()` → `optimizer.step()` → `optimizer.zero_grad()`，
  并把当前的 `X` 追加到记录中；
- 返回形状为 `(n_iters + 1, 2)` 的 numpy 数组（含初始点）。

然后用起点 `(-7.0, 2.0)`、`optim.SGD(lr=1e-2)`、迭代 500 步，调用刚写的函数跑一次，打印轨迹数组的形状和最后一个点。

In [ ]:
# 练习 3：编写通用的优化轨迹记录函数

# 练习 3：编写通用的优化轨迹记录函数

w = torch.tensor([[0.05], [1.0]], dtype=torch.float32)


def gradient_descent(X, optimizer, n_iters):
    X_arr = X.detach().numpy().copy()          # 记录初始点
    for _ in range(n_iters):
        y = X ** 2 @ w                         # 前向：目标函数值
        y.backward()                           # 反向：求梯度
        optimizer.step()                       # 更新参数
        optimizer.zero_grad()                  # 清空梯度
        X_arr = np.vstack([X_arr, X.detach().numpy()])   # 记录当前点
    return X_arr


X = torch.tensor([-7.0, 2.0], dtype=torch.float32, requires_grad=True)
arr = gradient_descent(X, optim.SGD([X], lr=1e-2), n_iters=500)

print("轨迹形状:", arr.shape)
print("最终点  :", arr[-1])

**练习 4**：SGD 的缺点——学习率不合适导致震荡或发散

目标函数 $f(x_1,x_2)=0.05x_1^{2}+x_2^{2}$ 中 $x_2$ 方向的曲率为 2，是更容易引起震荡的方向。

要求：复用练习 3 的 `gradient_descent`，从 `(-7.0, 2.0)` 出发，分别用学习率 `0.1`、`0.5`、`0.9`、`1.05`
迭代 300 步，打印每种学习率下的最终点和到最优解 `(0, 0)` 的距离（`np.linalg.norm`），并说明哪种学习率会发散。

In [ ]:
# 练习 4：SGD 的缺点：学习率不合适导致震荡或发散

# 练习 4：SGD 的缺点——学习率不合适导致震荡或发散

for lr in (0.1, 0.5, 0.9, 1.05):
    X = torch.tensor([-7.0, 2.0], dtype=torch.float32, requires_grad=True)
    arr = gradient_descent(X, optim.SGD([X], lr=lr), n_iters=300)
    print(f"lr={lr:<5} 最终点={arr[-1]} 到最优解的距离={np.linalg.norm(arr[-1]):.4e}")

**练习 5**：Momentum——用动量法减缓震荡、加快收敛

要求：在同一目标函数上，从同一起点 `(-7.0, 2.0)`、同样 `lr = 1e-2`、同样迭代 500 步，
分别用 `optim.SGD`（无动量）与 `optim.SGD(momentum=0.9)` 记录轨迹（复用练习 3 的 `gradient_descent`）：

1. 打印两条轨迹的最终点；
2. 在 `(-7.5, 2.5) × (-2.5, 2.5)` 范围内的等高线图上把两条轨迹画出来，并加图例（SGD / Momentum）。

In [ ]:
# 练习 5：Momentum：用动量法减缓震荡、加快收敛

# 练习 5：Momentum——用动量法减缓震荡、加快收敛

lr, n_iters = 1e-2, 500

X_sgd = torch.tensor([-7.0, 2.0], dtype=torch.float32, requires_grad=True)
arr_sgd = gradient_descent(X_sgd, optim.SGD([X_sgd], lr=lr), n_iters)

X_mom = torch.tensor([-7.0, 2.0], dtype=torch.float32, requires_grad=True)
arr_mom = gradient_descent(X_mom, optim.SGD([X_mom], lr=lr, momentum=0.9), n_iters)

print("SGD      最终点:", arr_sgd[-1])
print("Momentum 最终点:", arr_mom[-1])

x1_grid, x2_grid = np.meshgrid(np.linspace(-7.5, 2.5, 100), np.linspace(-2.5, 2.5, 100))
y_grid = 0.05 * x1_grid ** 2 + 1.0 * x2_grid ** 2

plt.contour(x1_grid, x2_grid, y_grid, levels=30, colors="gray")
plt.plot(arr_sgd[:, 0], arr_sgd[:, 1], "r-", label="SGD")
plt.plot(arr_mom[:, 0], arr_mom[:, 1], "b-", label="Momentum")
plt.legend()
plt.title("SGD vs Momentum")
plt.show()

**练习 6**：学习率衰减——等间隔衰减（StepLR）

要求：初始学习率 `lr0 = 0.9`，使用 `optim.lr_scheduler.StepLR(optimizer, step_size=20, gamma=0.7)`，
在目标函数上迭代 1000 次。

**注意顺序**：每次迭代先 `optimizer.step()`、再 `optimizer.zero_grad()`，然后记录
`optimizer.param_groups[0]["lr"]`，最后才调用 `scheduler.step()`。

1. 记录每一步的学习率；
2. 对 `epoch = 19, 20, 39, 40`，用公式 `lr0 * gamma ** (epoch // step_size)` 验证记录值（打印实际值与期望值）；
3. 画出学习率随 epoch 变化的曲线。

In [ ]:
# 练习 6：学习率等间隔衰减（StepLR）

# 练习 6：学习率衰减——等间隔衰减（StepLR）

X = torch.tensor([-7.0, 2.0], dtype=torch.float32, requires_grad=True)

lr0, n_iters, step_size, gamma = 0.9, 1000, 20, 0.7
optimizer = optim.SGD([X], lr=lr0)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=step_size, gamma=gamma)

lr_list = []
for epoch in range(n_iters):
    y = X ** 2 @ w
    y.backward()
    optimizer.step()
    optimizer.zero_grad()
    lr_list.append(optimizer.param_groups[0]["lr"])   # 先记录当前学习率
    scheduler.step()                                  # 再更新学习率

for epoch in (19, 20, 39, 40):
    decay_times = epoch // step_size                  # 到当前为止已衰减的次数
    expect = lr0 * gamma ** decay_times
    print(f"epoch={epoch:>3} 实际 lr={lr_list[epoch]:.6f} 期望 lr={expect:.6f}")

plt.plot(lr_list, "k-")
plt.xlabel("epoch")
plt.ylabel("lr")
plt.title("StepLR")
plt.show()

**练习 7**：学习率衰减——指定间隔衰减（MultiStepLR）

要求：初始学习率 `lr0 = 0.9`，使用 `optim.lr_scheduler.MultiStepLR(optimizer, milestones=[10, 50, 200], gamma=0.7)`，
迭代 400 次，记录顺序与练习 6 相同（先 `optimizer.step()` 并记录学习率，最后 `scheduler.step()`）。

1. 找出学习率发生变化的 epoch（对比相邻两个记录值）；
2. 对 `epoch = 9, 10, 49, 50, 199, 200`，用"已越过的里程碑个数"作为幂次验证记录值，
   即 `lr0 * gamma ** sum(1 for m in milestones if m <= epoch)`。

In [ ]:
# 练习 7：学习率指定间隔衰减（MultiStepLR）

# 练习 7：学习率衰减——指定间隔衰减（MultiStepLR）

X = torch.tensor([-7.0, 2.0], dtype=torch.float32, requires_grad=True)

lr0, n_iters, milestones, gamma = 0.9, 400, [10, 50, 200], 0.7
optimizer = optim.SGD([X], lr=lr0)
scheduler = optim.lr_scheduler.MultiStepLR(optimizer, milestones=milestones, gamma=gamma)

lr_list = []
for epoch in range(n_iters):
    y = X ** 2 @ w
    y.backward()
    optimizer.step()
    optimizer.zero_grad()
    lr_list.append(optimizer.param_groups[0]["lr"])
    scheduler.step()

changed = [i for i in range(1, n_iters) if lr_list[i] != lr_list[i - 1]]
print("学习率发生衰减的 epoch:", changed)

for epoch in (9, 10, 49, 50, 199, 200):
    passed = sum(1 for m in milestones if m <= epoch)
    expect = lr0 * gamma ** passed
    print(f"epoch={epoch:>3} 实际 lr={lr_list[epoch]:.6f} 期望 lr={expect:.6f}")

**练习 8**：学习率衰减——指数衰减（ExponentialLR）

要求：初始学习率 `lr0 = 0.9`，使用 `optim.lr_scheduler.ExponentialLR(optimizer, gamma=0.99)`，迭代 400 次，
记录顺序与练习 6 相同（先 `optimizer.step()` 并记录学习率，最后 `scheduler.step()`）。

1. 对 `epoch = 0, 1, 5, 20`，用公式 `lr0 * gamma ** epoch` 验证记录值；
2. 画出学习率随 epoch 变化的曲线（指数曲线的形态）。

In [ ]:
# 练习 8：学习率指数衰减（ExponentialLR）

# 练习 8：学习率衰减——指数衰减（ExponentialLR）

X = torch.tensor([-7.0, 2.0], dtype=torch.float32, requires_grad=True)

lr0, n_iters, gamma = 0.9, 400, 0.99
optimizer = optim.SGD([X], lr=lr0)
scheduler = optim.lr_scheduler.ExponentialLR(optimizer, gamma=gamma)

lr_list = []
for epoch in range(n_iters):
    y = X ** 2 @ w
    y.backward()
    optimizer.step()
    optimizer.zero_grad()
    lr_list.append(optimizer.param_groups[0]["lr"])
    scheduler.step()

for epoch in (0, 1, 5, 20):
    expect = lr0 * gamma ** epoch
    print(f"epoch={epoch:>3} 实际 lr={lr_list[epoch]:.6f} 期望 lr={expect:.6f}")

plt.plot(lr_list, "k-")
plt.xlabel("epoch")
plt.ylabel("lr")
plt.title("ExponentialLR")
plt.show()

**练习 9**：手写五种优化器的参数更新

要求：编写 `manual_optimizer(kind, lr, n_iters, alpha=0.9, betas=(0.9, 0.999), eps=1e-8)`，
从 `(-7.0, 2.0)` 出发手写实现下面 5 种更新规则，每一步记录 `X`，返回轨迹数组：

| kind | 更新规则 |
| --- | --- |
| `"sgd"` | $W \leftarrow W - \eta\nabla$ |
| `"momentum"` | $b \leftarrow \alpha b + \nabla,\quad W \leftarrow W - \eta b$ |
| `"adagrad"` | $h \leftarrow h + \nabla^{2},\quad W \leftarrow W - \eta\frac{\nabla}{\sqrt{h}+\varepsilon}$ |
| `"rmsprop"` | $h \leftarrow \alpha h + (1-\alpha)\nabla^{2},\quad W \leftarrow W - \eta\frac{\nabla}{\sqrt{h}+\varepsilon}$ |
| `"adam"` | $v \leftarrow \beta_1 v + (1-\beta_1)\nabla,\ h \leftarrow \beta_2 h + (1-\beta_2)\nabla^{2}$，偏差修正后 $W \leftarrow W - \eta\frac{\hat{v}}{\sqrt{\hat{h}}+\varepsilon}$ |

先用 `manual_optimizer("sgd", lr=1e-2, n_iters=10)` 打印最终点，验证函数可用。

提示：手写更新要在 `torch.no_grad()` 上下文里用 `X -= ...` 原地修改，然后 `X.grad.zero_()`；
`t` 从 1 开始计数，`adam` 的偏差修正为 `v / (1 - beta1 ** t)`、`h / (1 - beta2 ** t)`。

In [ ]:
# 练习 9：手写五种优化器的参数更新

# 练习 9：手写五种优化器的参数更新


def manual_optimizer(kind, lr, n_iters, alpha=0.9, betas=(0.9, 0.999), eps=1e-8):
    X = torch.tensor([-7.0, 2.0], dtype=torch.float32, requires_grad=True)
    buf = torch.zeros_like(X)      # 当前更新量
    h = torch.zeros_like(X)        # 二阶累积量
    v = torch.zeros_like(X)        # 一阶动量
    b1, b2 = betas
    step = 0
    arr = X.detach().numpy().copy()
    for _ in range(n_iters):
        y = X ** 2 @ w
        y.backward()
        grad = X.grad.detach().clone()
        step += 1

        if kind == "sgd":
            buf = grad
        elif kind == "momentum":
            buf = alpha * buf + grad
        elif kind == "adagrad":
            h = h + grad ** 2
            buf = grad / (torch.sqrt(h) + eps)
        elif kind == "rmsprop":
            h = alpha * h + (1 - alpha) * grad ** 2
            buf = grad / (torch.sqrt(h) + eps)
        elif kind == "adam":
            v = b1 * v + (1 - b1) * grad
            h = b2 * h + (1 - b2) * grad ** 2
            v_hat = v / (1 - b1 ** step)          # 偏差修正
            h_hat = h / (1 - b2 ** step)          # 偏差修正
            buf = v_hat / (torch.sqrt(h_hat) + eps)
        else:
            raise ValueError(f"未知的优化器: {kind}")

        with torch.no_grad():
            X -= lr * buf
        X.grad.zero_()
        arr = np.vstack([arr, X.detach().numpy()])
    return arr


print("sgd 迭代 10 步最终点:", manual_optimizer("sgd", lr=1e-2, n_iters=10)[-1])

**练习 10**：手写优化器与 torch.optim 逐一对比

要求：编写 `run_torch_opt(cls, lr, n_iters, **kwargs)`，用 `torch.optim` 中的优化器在同样的目标函数上迭代并返回轨迹，
然后与练习 9 的手写实现逐一对比（迭代 300 步，起点 `(-7.0, 2.0)`）：

| 优化器 | torch 参数 | 手写参数 |
| --- | --- | --- |
| SGD | `lr=1e-2` | `lr=1e-2` |
| Momentum | `lr=1e-2, momentum=0.9` | `lr=1e-2, alpha=0.9` |
| AdaGrad | `lr=0.9, eps=1e-10` | `lr=0.9, eps=1e-10` |
| RMSProp | `lr=0.1, alpha=0.99, eps=1e-8` | `lr=0.1, alpha=0.99, eps=1e-8` |
| Adam | `lr=0.1, betas=(0.9, 0.999), eps=1e-8` | `lr=0.1, betas=(0.9, 0.999), eps=1e-8` |

1. 打印每种优化器两条轨迹的最大绝对误差（应在 1e-6 以内）与各自的最终点；
2. 在同一张等高线图上画出 5 条轨迹并加图例。

提示：可以先用一个字典保存"名字 → (优化器类, torch 参数, 手写参数)"，再用循环统一处理。

In [ ]:
# 练习 10：手写优化器与 torch.optim 逐一对比

# 练习 10：手写优化器与 torch.optim 逐一对比


def run_torch_opt(cls, lr, n_iters, **kwargs):
    X = torch.tensor([-7.0, 2.0], dtype=torch.float32, requires_grad=True)
    optimizer = cls([X], lr=lr, **kwargs)
    arr = X.detach().numpy().copy()
    for _ in range(n_iters):
        y = X ** 2 @ w
        y.backward()
        optimizer.step()
        optimizer.zero_grad()
        arr = np.vstack([arr, X.detach().numpy()])
    return arr


settings = {
    "sgd":      (optim.SGD,     dict(lr=1e-2),                              dict(lr=1e-2)),
    "momentum": (optim.SGD,     dict(lr=1e-2, momentum=0.9),                dict(lr=1e-2, alpha=0.9)),
    "adagrad":  (optim.Adagrad, dict(lr=0.9, eps=1e-10),                    dict(lr=0.9, eps=1e-10)),
    "rmsprop":  (optim.RMSprop, dict(lr=0.1, alpha=0.99, eps=1e-8),         dict(lr=0.1, alpha=0.99, eps=1e-8)),
    "adam":     (optim.Adam,    dict(lr=0.1, betas=(0.9, 0.999), eps=1e-8), dict(lr=0.1, betas=(0.9, 0.999), eps=1e-8)),
}

x1_grid, x2_grid = np.meshgrid(np.linspace(-7.5, 2.5, 100), np.linspace(-2.5, 2.5, 100))
y_grid = 0.05 * x1_grid ** 2 + 1.0 * x2_grid ** 2
plt.contour(x1_grid, x2_grid, y_grid, levels=30, colors="gray")

for name, (cls, t_kwargs, m_kwargs) in settings.items():
    arr_t = run_torch_opt(cls, t_kwargs["lr"], 300,
                          **{k: v for k, v in t_kwargs.items() if k != "lr"})
    arr_m = manual_optimizer(name, m_kwargs["lr"], 300,
                             alpha=m_kwargs.get("alpha", 0.9),
                             betas=m_kwargs.get("betas", (0.9, 0.999)),
                             eps=m_kwargs.get("eps", 1e-8))
    print(f"{name:>9} torch 终点={arr_t[-1]} 手写终点={arr_m[-1]} "
          f"最大误差={np.abs(arr_t - arr_m).max():.3e}")
    plt.plot(arr_t[:, 0], arr_t[:, 1], label=name)

plt.legend()
plt.title("Manual vs torch.optim")
plt.show()

## 5.3 参数初始化

参数初始化的选择对数值稳定性至关重要：初始化不当会导致梯度消失或梯度爆炸。
PyTorch 中可以通过 `nn.init` 下的方法对参数进行初始化。

**练习 11**：常数初始化与秩初始化

要求：

1. 创建一个 `nn.Linear(3, 3)`，用 `nn.init.eye_` 初始化权重、`nn.init.zeros_` 初始化偏置，
   输入 `torch.tensor([[1.0, 2.0, 3.0]])`，验证输出与输入完全相同；
2. 演示"权重不能全部初始化为同一个值"：创建一个 `nn.Linear(4, 2)`，把权重和偏置全部置 0，
   输入 `torch.randn(3, 4)`，输出求和后反向传播，打印权重梯度并判断两行梯度是否完全相同。

提示：判断张量是否相同可以用 `torch.equal(a, b)`。

In [ ]:
# 练习 11：常数初始化与秩初始化

# 练习 11：常数初始化与秩初始化

# eye_ 初始化：权重为单位矩阵时，输出等于输入
linear = nn.Linear(3, 3)
nn.init.eye_(linear.weight)
nn.init.zeros_(linear.bias)

x = torch.tensor([[1.0, 2.0, 3.0]])
print("eye_ 输出:", linear(x).detach().numpy(), " 是否等于输入:", torch.allclose(linear(x), x))

# 全 0 初始化会保留"对称性"，导致同一层不同神经元的梯度完全一样
linear_zero = nn.Linear(4, 2)
nn.init.zeros_(linear_zero.weight)
nn.init.zeros_(linear_zero.bias)

out = linear_zero(torch.randn(3, 4))
out.sum().backward()
print("全 0 权重的梯度（两行应相同）:\n", linear_zero.weight.grad)
print("两行梯度是否相同:", torch.equal(linear_zero.weight.grad[0], linear_zero.weight.grad[1]))

**练习 12**：正态分布与均匀分布初始化

要求：创建一个 `nn.Linear(2000, 1000)`，分别进行下面的初始化，统计权重的均值、标准差（或最小/最大值）并与理论值对照：

1. `nn.init.normal_(weight, mean=1.0, std=0.3)`，理论均值 1.0、标准差 0.3；
2. `nn.init.uniform_(weight, a=2.0, b=5.0)`，理论区间 $[2.0, 5.0]$，标准差为 $\frac{b-a}{\sqrt{12}}$。

In [ ]:
# 练习 12：正态分布与均匀分布初始化

# 练习 12：正态分布与均匀分布初始化

linear = nn.Linear(2000, 1000)

nn.init.normal_(linear.weight, mean=1.0, std=0.3)
print("normal_  均值=%.4f 标准差=%.4f | 理论 1.0 / 0.3"
      % (linear.weight.mean().item(), linear.weight.std().item()))

nn.init.uniform_(linear.weight, a=2.0, b=5.0)
print("uniform_ 均值=%.4f 标准差=%.4f 最小值=%.4f 最大值=%.4f | 理论 3.5 / %.4f / 2.0 / 5.0"
      % (linear.weight.mean().item(), linear.weight.std().item(),
         linear.weight.min().item(), linear.weight.max().item(), 3.0 / (12 ** 0.5)))

**练习 13**：Xavier 初始化（Glorot 初始化）

Xavier 初始化根据输入数和输出数共同调整权重的初始范围，适用于 Sigmoid / Tanh 等激活函数：

- 正态分布：均值为 0，标准差为 $\sqrt{\frac{2}{n_{in}+n_{out}}}$
- 均匀分布：区间 $\left(-\sqrt{\frac{6}{n_{in}+n_{out}}},\ \sqrt{\frac{6}{n_{in}+n_{out}}}\right)$

要求：对 `nn.Linear(512, 256)` 分别调用 `nn.init.xavier_normal_` 与 `nn.init.xavier_uniform_`，
用统计出的标准差与理论值对照（正态分布看标准差，均匀分布看绝对值的最大值即边界）。

In [ ]:
# 练习 13：Xavier 初始化

# 练习 13：Xavier 初始化（Glorot 初始化）

n_in, n_out = 512, 256
linear = nn.Linear(n_in, n_out)

nn.init.xavier_normal_(linear.weight)
expect_std = (2 / (n_in + n_out)) ** 0.5
print("xavier_normal_  标准差=%.5f | 理论=%.5f"
      % (linear.weight.std().item(), expect_std))

nn.init.xavier_uniform_(linear.weight)
expect_bound = (6 / (n_in + n_out)) ** 0.5
print("xavier_uniform_ 标准差=%.5f | 理论=%.5f"
      % (linear.weight.std().item(), expect_std))
print("xavier_uniform_ 边界=%.5f   | 理论=%.5f"
      % (linear.weight.abs().max().item(), expect_bound))

**练习 14**：He 初始化（Kaiming 初始化）

He 初始化只根据输入数调整权重的初始范围，主要适用于 ReLU 及其变体：

- 正态分布：均值为 0，标准差为 $\sqrt{\frac{2}{n_{in}}}$
- 均匀分布：区间 $\left(-\sqrt{\frac{6}{n_{in}}},\ \sqrt{\frac{6}{n_{in}}}\right)$

要求：对 `nn.Linear(512, 256)` 分别调用 `nn.init.kaiming_normal_` 与 `nn.init.kaiming_uniform_`，
统计标准差与边界，并与理论值对照。

In [ ]:
# 练习 14：He（Kaiming）初始化

# 练习 14：He 初始化（Kaiming 初始化）

n_in, n_out = 512, 256
linear = nn.Linear(n_in, n_out)

nn.init.kaiming_normal_(linear.weight)
expect_std = (2 / n_in) ** 0.5
print("kaiming_normal_  标准差=%.5f | 理论=%.5f"
      % (linear.weight.std().item(), expect_std))

nn.init.kaiming_uniform_(linear.weight)
expect_bound = (6 / n_in) ** 0.5
print("kaiming_uniform_ 标准差=%.5f | 理论=%.5f"
      % (linear.weight.std().item(), expect_std))
print("kaiming_uniform_ 边界=%.5f   | 理论=%.5f"
      % (linear.weight.abs().max().item(), expect_bound))

## 5.4 正则化

过拟合指模型能较好拟合训练数据，却不能很好地预测训练数据之外的数据。常用的正则化方法有
Batch Normalization、权值衰减、Dropout、早停法等。

**练习 15**：BatchNorm1d——手写批量标准化并与 `nn.BatchNorm1d` 对比

Batch Normalization 先对每个特征在 batch 维度上做标准化，再做缩放和平移：

$$\mu=\frac{1}{n}\sum x, \qquad \sigma^{2}=\frac{1}{n}\sum (x-\mu)^{2}, \qquad
\hat{x}=\frac{x-\mu}{\sqrt{\sigma^{2}+\epsilon}}, \qquad y=\gamma\hat{x}+\beta$$

要求：

1. 创建 `nn.BatchNorm1d(num_features=3)`，先 `torch.manual_seed(0)`，输入 `torch.randn(5, 3)`，得到 BN 的输出；
2. 手写上述公式（方差用有偏估计，即 `x.var(dim=0, unbiased=False)`；`gamma`、`beta` 用 BN 层自带的
   `weight`、`bias`，`epsilon` 用 `bn.eps`），并与 BN 层的输出比较最大绝对误差；
3. 打印训练模式下 BN 输出在每个特征方向上的均值（应接近 0）；
4. 调用 `bn.eval()` 后再前向一次并打印 `running_mean`，体会训练 / 评估两种模式的区别。

In [ ]:
# 练习 15：BatchNorm1d：手写批量标准化并与 nn.BatchNorm1d 对比

# 练习 15：BatchNorm1d——手写批量标准化并与 nn.BatchNorm1d 对比

torch.manual_seed(0)
bn = nn.BatchNorm1d(num_features=3)
x = torch.randn(5, 3)

y = bn(x)                                     # BatchNorm 输出（训练模式）

mu = x.mean(dim=0)                            # 每个特征的均值
var = x.var(dim=0, unbiased=False)            # 每个特征的方差（有偏）
x_hat = (x - mu) / torch.sqrt(var + bn.eps)   # 标准化
y_manual = bn.weight * x_hat + bn.bias        # 缩放平移

print("最大绝对误差:", (y - y_manual).abs().max().item())
print("两者是否一致:", torch.allclose(y, y_manual, atol=1e-6))
print("训练模式下 BN 输出的特征均值:", y.mean(dim=0).detach().numpy())

# 评估模式：使用训练阶段累计的 running_mean / running_var
bn.eval()
print("评估模式输出:\n", bn(x).detach().numpy())
print("running_mean:", bn.running_mean.numpy())

**练习 16**：权值衰减

权值衰减通过在损失函数上加入 L2 惩罚项 $L' = L + \frac{1}{2}\lambda\|W\|^{2}$ 抑制过拟合。
惩罚项求导得到 $\lambda W$，因此等价于在梯度上额外加上 $\lambda W$。

要求：设权重初值 `(2.0, -1.0)`、损失取 $L = \|W\|^{2}$（因此梯度 `grad = 2 * W`）、
`lr = 0.1`、`lam = 0.02`，迭代 20 步：

1. 手写更新：`W = W - lr * (grad + lam * W)`；
2. 用 `optim.SGD(lr=lr, weight_decay=lam)` 做同样的迭代（每步手动设置 `param.grad = 2 * param.detach()`）；
3. 比较两者结果是否一致。

In [ ]:
# 练习 16：权值衰减（weight_decay）

# 练习 16：权值衰减

lr, lam, n_iters = 0.1, 0.02, 20

# 手写：L2 惩罚项求导得到 lam * W，额外加到梯度上
q = torch.tensor([2.0, -1.0])
for _ in range(n_iters):
    grad = 2 * q                                  # L = ||W||^2 的梯度
    q = q - lr * (grad + lam * q)

# torch：通过 weight_decay 实现同样的效果
param = nn.Parameter(torch.tensor([2.0, -1.0]))
optimizer = optim.SGD([param], lr=lr, weight_decay=lam)
for _ in range(n_iters):
    param.grad = 2 * param.detach().clone()
    optimizer.step()
    optimizer.zero_grad()

print("手写结果  :", q.numpy())
print("weight_decay:", param.detach().numpy())
print("两者是否一致:", torch.allclose(q, param.detach(), atol=1e-6))

**练习 17**：Dropout——手写随机失活并与 `nn.Dropout` 对比

训练时以概率 $p$ 随机关闭神经元，未被关闭的神经元输出按 $\frac{1}{1-p}$ 缩放，使期望值不变；
测试时不使用 Dropout。

要求：

1. 编写 `my_dropout(x, p, training=True)`：用 `torch.rand_like(x) > p` 生成掩码，返回 `x * mask / (1 - p)`；
   当 `training=False` 或 `p == 0` 时直接返回 `x`；
2. 先 `torch.manual_seed(0)`，用全 1 输入 `torch.ones(1000, 5)`、`p = 0.3`：打印失活比例与非零元素的均值，
   并与理论值（0.3、1/(1-0.3)）对照；
3. 验证 `training=False` 以及 `p = 0` 时输出与输入完全相同。

In [ ]:
# 练习 17：Dropout：手写随机失活并与 nn.Dropout 对比

# 练习 17：Dropout——手写随机失活并与 nn.Dropout 对比


def my_dropout(x, p, training=True):
    if (not training) or p == 0:
        return x
    mask = (torch.rand_like(x) > p).float()       # 以概率 p 置 0
    return x * mask / (1 - p)                     # 缩放，保持期望不变


torch.manual_seed(0)
x = torch.ones(1000, 5)
p = 0.3

y_manual = my_dropout(x, p)
print("手写      失活比例=%.4f 非零均值=%.4f | 理论 %.1f / %.4f"
      % ((y_manual == 0).float().mean().item(),
         y_manual[y_manual != 0].mean().item(), p, 1 / (1 - p)))

torch.manual_seed(0)
y_torch = nn.Dropout(p)(x)
print("nn.Dropout 失活比例=%.4f 非零均值=%.4f"
      % ((y_torch == 0).float().mean().item(),
         y_torch[y_torch != 0].mean().item()))

print("评估模式下与输入相同:", torch.equal(my_dropout(x, p, training=False), x))
print("p=0 时与输入相同    :", torch.equal(my_dropout(x, 0.0), x))

## 5.5 应用案例：房价预测

使用 House Prices 数据集完成一个完整的回归流程：特征工程 → 搭建模型 → 定义损失函数 → 训练模型。

数据位于 `../../data/house_prices.csv`，目标列是 `SalePrice`，其中既有数值型特征也有类别型特征，
并且存在缺失值，需要分别处理。

**练习 18**：特征工程——构造数据集

要求：编写 `create_dataset()`，返回 `(train_dataset, test_dataset, feature_num)`：

1. 用 `pd.read_csv` 读取 `../../data/house_prices.csv`，用 `drop` 去掉无关特征 `Id`；
2. 划分特征 `X`（去掉 `SalePrice`）与目标 `y`（`SalePrice`）；
3. 用 `select_dtypes(exclude="object")` 与 `select_dtypes(include="object")` 分别筛出数值型、类别型特征；
4. 用 `train_test_split` 按 `test_size=0.2, random_state=42` 划分训练集与测试集；
5. 数值型特征：`SimpleImputer(strategy="mean")` 填充缺失值 + `StandardScaler()` 标准化；
   类别型特征：`SimpleImputer(strategy="constant", fill_value="NaN")` 填充 + `OneHotEncoder(handle_unknown="ignore")` 独热编码；
   两者用 `ColumnTransformer` 组合，`fit_transform` 训练集、`transform` 测试集（注意转成 `DataFrame` 并取列名）；
6. 用 `TensorDataset` 把特征与目标包装成 float32 张量，并返回特征数量。

最后调用一次，打印特征数量与训练 / 测试集大小。

In [ ]:
# 练习 18：特征工程：构造数据集

# 练习 18：特征工程——构造数据集

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder


def create_dataset():
    # 1. 读取数据并去掉无关特征
    data = pd.read_csv("../../data/house_prices.csv")
    data.drop(["Id"], axis=1, inplace=True)

    # 2. 划分特征与目标
    X = data.drop("SalePrice", axis=1)
    y = data["SalePrice"]

    # 3. 数值型 / 类别型特征
    numerical_features = X.select_dtypes(exclude="object").columns
    categorical_features = X.select_dtypes(include="object").columns

    # 4. 划分训练集与测试集
    x_train, x_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42)

    # 5. 特征预处理
    numerical_transformer = Pipeline(steps=[
        ("fillna", SimpleImputer(strategy="mean")),
        ("std", StandardScaler()),
    ])
    categorical_transformer = Pipeline(steps=[
        ("fillna", SimpleImputer(strategy="constant", fill_value="NaN")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ])
    preprocessor = ColumnTransformer(transformers=[
        ("num", numerical_transformer, numerical_features),
        ("cat", categorical_transformer, categorical_features),
    ])
    x_train = pd.DataFrame(preprocessor.fit_transform(x_train).toarray(),
                           columns=preprocessor.get_feature_names_out())
    x_test = pd.DataFrame(preprocessor.transform(x_test).toarray(),
                          columns=preprocessor.get_feature_names_out())

    # 6. 构造 TensorDataset
    train_dataset = TensorDataset(torch.tensor(x_train.values).float(),
                                  torch.tensor(y_train.values).float())
    test_dataset = TensorDataset(torch.tensor(x_test.values).float(),
                                 torch.tensor(y_test.values).float())
    return train_dataset, test_dataset, x_train.shape[1]


train_dataset, test_dataset, feature_num = create_dataset()
print("特征数量:", feature_num)
print("训练集大小:", len(train_dataset), " 测试集大小:", len(test_dataset))

**练习 19**：搭建模型与损失函数

要求：

1. 用 `nn.Sequential` 搭建回归模型，结构为
   `Linear(feature_num, 128) → BatchNorm1d(128) → ReLU → Dropout(0.2) → Linear(128, 1)`，并打印模型结构；
2. 实现对数均方根误差损失（房价预测更关心相对误差）：

$$Loss=\sqrt{\frac{1}{n}\sum_{i=1}^{n}\left(\log(\hat{y})-\log(y)\right)^{2}}$$

   注意把预测值 `squeeze` 后用 `torch.clamp(pred, 1, float("inf"))` 限制在 1 到正无穷之间（避免 `log` 出现非法值）；
3. 用 `pred = torch.tensor([100.0, 200.0])`、`target = torch.tensor([110.0, 180.0])` 验证损失函数能正常计算。

In [ ]:
# 练习 19：搭建模型与损失函数

# 练习 19：搭建模型与损失函数

model = nn.Sequential(
    nn.Linear(feature_num, 128),
    nn.BatchNorm1d(128),
    nn.ReLU(),
    nn.Dropout(0.2),
    nn.Linear(128, 1),
)
print(model)


def log_rmse(pred, target):
    mse = nn.MSELoss()
    pred = pred.squeeze()
    pred = torch.clamp(pred, 1, float("inf"))     # 限制输出在 1 到正无穷之间
    return torch.sqrt(mse(torch.log(pred), torch.log(target)))


demo_pred = torch.tensor([100.0, 200.0])
demo_target = torch.tensor([110.0, 180.0])
print("log_rmse:", log_rmse(demo_pred, demo_target).item())

**练习 20**：模型训练与损失曲线

要求：编写 `train(model, train_dataset, test_dataset, lr, epoch_num, batch_size, device)`：

1. 用一个 `init_weight(layer)` 函数对线性层的权重做 `nn.init.xavier_normal_(layer.weight)` 初始化，
   通过 `model.apply(init_weight)` 应用；
2. 优化器使用 `torch.optim.Adam(model.parameters(), lr=lr)`；
3. 每个 epoch：`model.train()` + `DataLoader(shuffle=True)` 训练，累加并记录平均 `log_rmse`；
   然后 `model.eval()` + `torch.no_grad()` 在测试集上（`shuffle=False`）评估并记录平均 `log_rmse`；
4. 每 10 个 epoch 打印一次 train / test 损失，最后返回两个损失列表；
5. 用 `lr=0.1`、`epoch_num=100`、`batch_size=64` 训练，并把两条损失曲线画在同一张图上。

提示：设备用 `torch.device("cuda" if torch.cuda.is_available() else "cpu")`；
`log_rmse` 使用练习 19 中定义的函数。

In [ ]:
# 练习 20：模型训练与损失曲线

# 练习 20：模型训练与损失曲线


def train(model, train_dataset, test_dataset, lr, epoch_num, batch_size, device):
    def init_weight(layer):
        # 对线性层的权重做 Xavier 初始化
        if isinstance(layer, nn.Linear):
            nn.init.xavier_normal_(layer.weight)

    model.apply(init_weight)
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    train_loss_list = []
    test_loss_list = []
    for epoch in range(epoch_num):
        # 训练
        model.train()
        train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
        loss_accumulate = 0.0
        for X, y in train_loader:
            X, y = X.to(device), y.to(device)
            loss_value = log_rmse(model(X), y)
            optimizer.zero_grad()
            loss_value.backward()
            optimizer.step()
            loss_accumulate += loss_value.item()
        train_loss_list.append(loss_accumulate / len(train_loader))

        # 验证
        model.eval()
        test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
        loss_accumulate = 0.0
        with torch.no_grad():
            for X, y in test_loader:
                X, y = X.to(device), y.to(device)
                loss_accumulate += log_rmse(model(X), y).item()
        test_loss_list.append(loss_accumulate / len(test_loader))

        if (epoch + 1) % 10 == 0 or epoch == 0:
            print(f"epoch {epoch + 1:>3}/{epoch_num} train_loss:{train_loss_list[-1]:.4f} "
                  f"test_loss:{test_loss_list[-1]:.4f}")

    return train_loss_list, test_loss_list


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(0)
train_loss_list, test_loss_list = train(model, train_dataset, test_dataset,
                                        lr=0.1, epoch_num=100, batch_size=64, device=device)

plt.plot(train_loss_list, "r-", label="train_loss")
plt.plot(test_loss_list, "k--", label="test_loss")
plt.xlabel("epoch")
plt.ylabel("log rmse")
plt.legend()
plt.show()